# 10_01 Support answers with citations, and when to hand over

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: rung 2, on a savings firm's documents
>
> Project files this connects to:
>
> - `src/homer_ai/domain.py`
> - `src/homer_ai/domain_server.py`
> - `data/domain/savings/`
> - `scripts/benchmark_domain.py`
>
> **This notebook:** A customer asks about a savings account. How do we answer from the product sheet with its section and version, get a rate exactly right, and hand over to a person when the assistant shouldn't answer?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to cut product documents into passages that carry a version and a date
- Why a search is the wrong tool for an exact number, and what a lookup tool does instead
- When an assistant must not answer, and two layers that make it hand over
- How to score it: right answer, right source, invented facts, and handovers
- What one run of Claude with and without the three tools showed

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **product sheet** | the document that sets out how one product works: who can have it, limits, notice, interest | `easy_access.md`, 7 numbered sections |
| **AER** | annual equivalent rate: the interest a savings account pays over a year, in percent | 2.85% AER |
| **lookup tool** | a tool that returns one exact value from a table, where a search returns passages to read | `get_rate("easy-access", "2026-10-01")` |
| **guard** | a simple check that runs beside the model and can overrule it | a list of words that always sends a message to a person |
| **audit trail** | a record of what a system did, kept so that somebody can check it later | one line per tool call: when, which tool, which data, what came back |

## 3. The idea

### ❓ What are we building?

The support desk of Example Bank, a savings firm I made up. Everything here is invented: the firm, its products, its customers and their words ([data/domain/README.md](../../data/domain/README.md)).

Think of a new colleague on their first day at that desk. They get three rules.

1. Look it up in the product sheet, and tell the customer which section you read.
2. Never quote a rate from memory. Read it off the rates table, for the right day.
3. If the customer is upset, unwell, grieving, or asking what they should do with their money, don't answer. Fetch a senior colleague.

Three rules, three tools: `search_terms`, `get_rate` and `escalate`. The colleague is a big model like Claude, and the tools reach it over MCP, the way `search_homer` did in [09_03](../09_advanced_grounded_answers/09_03_the_search_tool.ipynb). This is rung 2 of the ladder: nothing is trained.

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    INK,
    MUTED,
    arrow,
    box,
    diagram_axes,
    studio_figure,
    tag,
)

fig = studio_figure("Support desk", "Look it up, or hand it over", (10, 6.8))
ax = diagram_axes(fig)

box(ax, 1.5, 5.15, 2.5, 0.9, "a customer's\nmessage", INK, fontsize=10)
box(
    ax,
    5.0,
    5.15,
    3.0,
    0.9,
    "the assistant\n(a big model)",
    ACCENT,
    fontsize=10.5,
    lw=2.2,
)
arrow(ax, (2.75, 5.15), (3.5, 5.15), color=MUTED, width=2)
tag(ax, 8.3, 5.15, "three tools,\nover MCP", MUTED, 9)

box(
    ax,
    1.9,
    3.45,
    2.9,
    0.95,
    "search_terms\nthe product sheets",
    GOOD,
    mono=True,
    fontsize=9.5,
)
box(
    ax, 5.0, 3.45, 2.9, 0.95, "get_rate\nthe rates table", GOOD, mono=True, fontsize=9.5
)
box(
    ax,
    8.1,
    3.45,
    2.9,
    0.95,
    "escalate\ncomplaint, vulnerable,\nadvice, bereavement",
    BAD,
    mono=True,
    fontsize=8.5,
)
arrow(ax, (4.3, 4.7), (2.4, 3.95), color=GOOD)
arrow(ax, (5.0, 4.7), (5.0, 3.95), color=GOOD)
arrow(ax, (5.7, 4.7), (7.6, 3.95), color=BAD)

box(
    ax,
    3.45,
    1.7,
    5.4,
    0.95,
    'an answer with its source\n"2.85%"  [rates#easy-access/2026-09-01]',
    GOOD,
    mono=True,
    fontsize=9.5,
    lw=2.2,
)
box(
    ax, 8.1, 1.7, 2.9, 0.95, "no answer:\na person takes over", BAD, fontsize=10, lw=2.2
)
arrow(ax, (1.9, 2.95), (2.6, 2.2), color=GOOD, width=2)
arrow(ax, (5.0, 2.95), (4.3, 2.2), color=GOOD, width=2)
arrow(ax, (8.1, 2.95), (8.1, 2.2), color=BAD, width=2)

box(
    ax,
    5.0,
    0.4,
    9.2,
    0.55,
    "every tool call adds one line to the audit trail",
    MUTED,
    fontsize=9.5,
    dashed=True,
)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A message comes in at the top. The assistant picks a tool. Two of them lead to an answer with its source in square brackets. The third leads to no answer at all: a person takes over.

The dashed strip at the bottom is the part people forget. Every tool call leaves one line behind, so that later somebody can see what the system did.

</details>

### ❓ Why can't a big model just answer?

Because it has never read these documents. Nobody has: I made them up for this chapter. Ask a model what Example Bank's notice period is, and the best it can do is say *I don't know*. The worst it can do is give you a plausible number.

That's the difference from chapter 9. Claude had read Homer, so the closed-book exam went well. A company's own documents are the case where there's no closed-book option, and at the end of this notebook we measure exactly that.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what do the documents look like?

Four product sheets and a FAQ, as Markdown files. Each starts with a small head, then numbered sections.

👉 **Predict first:** the five files have 7, 8, 7, 7 and 10 sections. If one section is one passage, how many passages is that?

In [ ]:
from homer_ai.domain import SAVINGS, load_sections

print(
    (SAVINGS / "notice_95_day.md").read_text(encoding="utf-8").split("## 2.")[0] + "…\n"
)

sections = load_sections()
print(f"{'sheet':14} {'sections':>8} {'version':>8}   in force from")
for doc in dict.fromkeys(s.doc for s in sections):
    mine = [s for s in sections if s.doc == doc]
    print(f"{doc:14} {len(mine):>8} {mine[0].version:>8}   {mine[0].in_force_from}")
print(f"{'all':14} {len(sections):>8}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

39 passages. That's tiny next to the 2,140 of chapter 9, and it's about the size of a real product range.

Look at the head of the sheet. `Version: 4` and `In force from: 2026-09-01` say which edition this is and since when it applies. `load_sections` in `domain.py` reads those lines and attaches them to every section.

</details>

### ❓ Step 2: what is a passage here, and why does it carry a date?

In chapter 9 we cut the poems by lines, because that's all the structure they had. A product sheet has headings, written by the people who know where one topic ends. So we cut along them: one numbered section is one passage.

```
ref           version  in force from  text
notice-95#4         4  2026-09-01     To take money out you give us 95 days' notice, …
```

The reference `notice-95#4` reads: the Notice Saver's sheet, section 4. That's the citation.

The version and the date are there because product sheets change. Homer doesn't. If the notice period was 90 days until August and 95 from September, then *which sheet did the answer come from?* is half of the answer. An assistant that quotes a rule with no date on it is quoting something that may have stopped being true.

In [ ]:
section = next(s for s in sections if s.ref == "notice-95#4")
print(f"{'ref':13} {'version':>7}  {'in force from':13}  text")
print(
    f"{section.ref:13} {section.version:>7}  {section.in_force_from:13}  "
    f"{section.text[:48]}…"
)
print()
print(f"shown to the model as: [{section.ref}] ({section.stamp})")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same row as in the picture, and the form the model will see: the reference in square brackets, then the version and its date.

</details>

### ❓ Step 3: does the search of chapter 9 work on this?

`search_terms` is the `BM25` class of [09_01](../09_advanced_grounded_answers/09_01_chunking_and_bm25.ipynb), unchanged, over the 39 sections. Let's ask it something a customer would.

The answer is in `easy-access#4`: *ask after 3pm and it arrives the next working day*.

👉 **Predict first:** will that section come first?

In [ ]:
from homer_ai.domain import search_terms

question = "If I ask for an Easy Access withdrawal at 4pm, when does the money arrive?"
for section, score in search_terms(question, k=3):
    print(f"{section.ref:15} {score:5.1f}   {section.doc_title}: {section.title}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Second, not first. The winner is `fixed-1-year#6`, which is about the end of a fixed term. Read it and you'll see why: it says *ask*, *money* and *Easy Access Saver* (that's where a matured fixed saver goes). The search counts words, and that section has the customer's words.

It's the lesson of 09_02 again. One passage isn't enough. We hand the model three, and it reads them.

</details>

### ❓ Step 4: how do I help the search tell four products apart?

Here's a problem Homer didn't have. Section 4 of three different sheets is called *Withdrawals*, and each says roughly the same things in the same words. Inside the section, nothing says which product it belongs to. That's written once, at the top of the file.

So we index each section with its sheet's title in front:

```
without:  You can take money out at any time, with no notice and no charge. …
with:     Easy Access Saver. Withdrawals. You can take money out at any time, …
```

Is that worth it? We don't guess. `questions.jsonl` has 22 questions whose answer is in one known section, so we can count, as in 09_02: **recall@k** is the share of questions whose right section is among the first k.

👉 **Predict first:** without the titles, for how many of the 22 is the right section first?

In [ ]:
from homer_ai.domain import load_support_questions
from homer_ai.retrieval import retrieval_scores

questions = load_support_questions()
terms = [q for q in questions if q["kind"] == "terms"]


def first_right(question, with_titles):
    found = search_terms(question["question"], k=5, with_titles=with_titles)
    for rank, (section, _score) in enumerate(found, start=1):
        if section.ref in question["refs"]:
            return rank
    return None


print(f"{len(terms)} questions{'':14} {'recall@1':>8} {'recall@3':>8}")
for name, with_titles in (
    ("the section alone", False),
    ("with the sheet's title", True),
):
    scores = retrieval_scores([first_right(q, with_titles) for q in terms], ks=(1, 3))
    print(f"{name:26} {scores['recall@1']:>8.3f} {scores['recall@3']:>8.3f}")

print("\nstill not in the first 3:")
for q in terms:
    rank = first_right(q, True)
    if rank is None or rank > 3:
        print(f"  {q['id']:>2}  rank {rank}   {q['question']}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| | right section first | in the first 3 |
|---|---|---|
| the section alone | 10 of 22 (0.455) | 15 of 22 (0.682) |
| with the sheet's title | 15 of 22 (0.682) | 19 of 22 (0.864) |

Two short titles in front of each passage, and four more questions find their section. It's the cheapest fix in retrieval, and it's easy to miss because the text looks complete to a person who knows which file it came from.

Three questions still miss, and each one is the weakness we know. Question 2 says *take money out before the year is up*; the sheet says *no withdrawals during the 12-month term*. Same meaning, no shared word. Question 15 says *warning* and *goes down* where the sheet says *notice* and *lower*. Question 11 finds the right topic in the wrong product.

With 22 questions, one question is 4.5 points. So read this as *about five in six*, no finer.

</details>

### ❓ Step 5: what's the interest rate? Why the search can't say

Ask the search for a rate and see what comes back.

👉 **Predict first:** will any of the three passages contain a percentage?

In [ ]:
from homer_ai.domain import load_rates

rate_question = "What rate does the Easy Access Saver pay on 1 October 2026?"
for section, _score in search_terms(rate_question, k=3):
    print(f"[{section.ref}] {section.text[:190]}…")

print()
previous = None
for rate in load_rates():
    name = rate.product if rate.product != previous else ""
    print(f"{name:14} from {rate.in_force_from}   {rate.aer:.2f}%")
    previous = rate.product

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

No percentage anywhere. The third passage even says so: *current and past rates are in the rates table, not in this sheet*. That's on purpose. Rates change far more often than terms do, and a number that lives in one place only has to be changed in one place.

The table underneath is that place: `rates.csv`, two rows per product. The Easy Access rate was 3.10% from March and is 2.85% from 1 September.

So *what's the rate?* has no answer until you also ask *on which day?* A search can't do that. It finds text that looks like the question. Picking the right row for a date is a different kind of job.

</details>

### ❓ Step 6: how does a lookup tool answer it?

`get_rate(product, on)` takes the rows of one product, keeps those that started on or before the day, and returns the last of them.

```
easy-access   from 2026-03-01  3.10%   ← in force on 31 August
              from 2026-09-01  2.85%   ← in force from 1 September on
```

The model's part shrinks to choosing two arguments. The number itself never passes through its memory.

And when there's no row, the tool says so. It doesn't take the nearest one.

👉 **Predict first:** the table starts on 1 March 2026. What should `get_rate` do for 1 February?

In [ ]:
from homer_ai.domain import get_rate

for day in ("2026-08-31", "2026-09-01", "2026-11-24"):
    rate = get_rate("easy-access", day)
    print(f"easy-access on {day}: {rate.aer:.2f}%   [{rate.ref}]")
print()
for day in ("2026-08-01", "2026-08-20"):
    rate = get_rate("fixed-1-year", day)
    print(f"fixed saver opened on {day}: {rate.aer:.2f}%   [{rate.ref}]")
print()
tries = (
    ("easy-access", "2026-02-01"),
    ("current-account", "2026-10-01"),
    ("easy-access", "1 October"),
)
for product, day in tries:
    try:
        get_rate(product, day)
    except (ValueError, LookupError) as error:
        print(f"{type(error).__name__}: {error}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

31 August gives 3.10% and 1 September gives 2.85%: the row changes on the day it says. Each result carries its own reference, like `rates#easy-access/2026-09-01`, which is the row it was read from.

The two fixed-saver lines are the trap in this product. A fixed rate is the one in force on the day the account **opened**. Two customers who opened three weeks apart get 4.20% and 4.05%, and both are right. The tool's description tells the model to ask for that day.

The last three lines are refusals: no rate that early, no such product, not a date. For a number that goes to a customer, *I can't find it* is a correct answer, and a guess never is.

</details>

### ❓ Step 7: when must the assistant not answer at all?

Some messages shouldn't get an answer from a program, however good the documents are. This invented firm has four kinds:

| Reason | A message like | Why a person takes it |
|---|---|---|
| **complaint** | *This is the third time I have asked for my statement and still nothing.* | a complaint has to be recorded and handled by somebody who can put it right |
| **vulnerability** | *I lost my job last month and I can't pay my rent. Can I break my fixed saver?* | the right reply depends on the person's situation, and a rule read out flat can do harm |
| **advice** | *Should I choose the fixed saver or the ISA?* | explaining a product is fine; telling someone what to do with their money is advice, and in many countries only authorised people may give it |
| **bereavement** | *My mother passed away last week and she had an account with you.* | nobody should get a bot's summary at that moment |

Two things about that table. The complaint doesn't contain the word *complaint*. As I understand rules like the UK's complaint-handling rules, what counts is that the customer is expressing dissatisfaction, in whatever words. I haven't checked that against the rule text for this course, so read the rules yourself before you rely on it. And the bereavement message does have an answer in the documents, in `faq#6`. The firm still wants a person on it. Which messages go to people is the firm's decision. It isn't the model's, and it isn't mine.

`escalate(reason)` is the tool for all four. In this course it only returns a record; a real one would open a case.

In [ ]:
from homer_ai.domain import escalate

for key, value in escalate("bereavement").items():
    print(f"{key:7} {value}")
print()
try:
    escalate("too difficult")
except ValueError as error:
    print("ValueError:", error)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The handover names the reason, the team that picks it up, and the one sentence the assistant may say. Then it stops.

The reason has to be one of four words. *Too difficult* is refused, so the model can't use the tool as a way out of an ordinary question without saying which of the four it is.

</details>

### ❓ Step 8: who decides to hand over, the model or a rule?

Both, in two layers.

The model reads the description of `escalate` and decides, message by message. That's flexible, and it's also a judgement you can't fully predict.

So under it sits a **guard**: `needs_person`, a few lists of words (*complain*, *passed away*, *diagnosed*, *should I*…). If a message contains one, it goes to a person whatever the model thinks. A guard is crude, but it does the same thing every time, and nobody can talk it out of it.

How much does the crude layer catch? `questions.jsonl` has 10 messages that must be handed over. The score for that is **recall**: of the messages that should have been caught, the share that were.

👉 **Predict first:** how many of the 10 will a word list catch? And will it raise a false alarm on any of the 29 ordinary questions?

In [ ]:
from homer_ai.domain import needs_person

must = [q for q in questions if q["kind"] == "escalate"]
others = [q for q in questions if q["kind"] != "escalate"]

caught = 0
for q in must:
    reason = needs_person(q["question"])
    caught += reason is not None
    print(
        f"{q['reason']:14} {'caught' if reason else 'MISSED':7} {q['question'][:60]}…"
    )
false_alarms = sum(needs_person(q["question"]) is not None for q in others)
print(f"\ncaught {caught} of {len(must)}: recall {caught / len(must):.2f}")
print(f"false alarms on the {len(others)} ordinary questions: {false_alarms}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

7 of 10, so a recall of 0.70, and no false alarm.

Look at the three it missed. Two are complaints without a complaint word: *this is the third time I have asked* and *I have had enough*. The third mentions a stroke, which isn't on my list. Each time I'd add a word, a new message would get past it.

That's the limit of a word list, and it's why the guard is the safety net and not the whole act. It catches the plain cases for certain. The quiet ones need something that reads the sentence. Here that's the big model. In [10_02](10_02_spotting_complaints.ipynb) we train a small one for the job.

</details>

### ❓ Step 9: how do the three tools reach an AI app?

Through a second MCP server, `src/homer_ai/domain_server.py`. It's built like the one of [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb): three functions with `@mcp.tool()` on top, and each docstring is the description the model reads.

Why not add them to `homer-serve`? Because a server is also a boundary. An app that helps with savings questions should see three savings tools and nothing else, and it has no business with a Greek model.

Let's look at what a model is told, without starting anything.

In [ ]:
from homer_ai import domain_server

tools = {tool.name: tool for tool in await domain_server.mcp.list_tools()}
for name, tool in tools.items():
    print(f"{name:13} {tool.input_schema['properties']}")
print()
print(tools["escalate"].description)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three tools. `product` and `reason` are an `enum`, a fixed list of allowed values, as `poem` was in 09_03: the model can't ask for a product that doesn't exist.

Read the description of `escalate`. It names the four cases in plain words, it says a complaint needs no special word, and it ends with a rule for the unclear case: *when in doubt, escalate*. That last line is a choice about which mistake is cheaper. A person can hand an easy question back. A wrong answer to somebody in trouble can't be unsent.

</details>

### ❓ What does a call return?

Text, with the reference in square brackets, ready to be quoted.

In [ ]:
print(domain_server.search_terms("Easy Access withdrawal after 3pm", k=1))
print()
print(domain_server.get_rate("fixed-1-year", "2026-08-20"))
print()
print(domain_server.escalate("complaint"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A section with its reference and version, a rate with the row it came from, and a handover that tells the model to stop.

The lines starting `domain: {…}` are the server's log. Like `homer-serve`, it writes them to stderr, because stdout carries the MCP messages.

To use the server from Claude Code, it's the pattern of 06_03 with another name and command:

```
claude mcp add example-bank -- uv --directory /path/to/homer-ai run python -m homer_ai.domain_server
```

The settings for the other apps are those of [09_03](../09_advanced_grounded_answers/09_03_the_search_tool.ipynb), with `python -m homer_ai.domain_server` in place of `homer-serve`.

</details>

### ❓ Step 10: what does the audit trail hold?

Those log lines are more than debugging. Set the environment variable `HOMER_DOMAIN_AUDIT` to a file name, and the server also adds each line to that file. That file is the **audit trail**.

👉 **Predict first:** we search for a customer's question. Will the question's text be in the file?

In [ ]:
import os
import tempfile
from pathlib import Path

audit_file = Path(tempfile.mkdtemp()) / "audit.jsonl"
os.environ["HOMER_DOMAIN_AUDIT"] = str(audit_file)
domain_server.search_terms("How long does an ISA transfer take?")
domain_server.get_rate("cash-isa", "2026-05-01")
domain_server.escalate("vulnerability")
del os.environ["HOMER_DOMAIN_AUDIT"]

print(audit_file.read_text(encoding="utf-8"))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Three calls, three lines. Each has the time, the tool, and what came back: the sections found, the rate row, the team.

`data` is a fingerprint of the product sheets and the rates table together. If anybody edits one of them, it changes. So a line doesn't only say *section 5 of the ISA sheet*. It says section 5 of the sheets as they were that day.

And the customer's words aren't there, only `query_chars`, their length. That's a choice. A log of everything customers type is itself a pile of personal data that somebody has to protect. A real firm decides what it must keep (usually more than this: who asked, which model, the full reply) and for how long. I'd rather the default kept too little than too much.

</details>

### ❓ Step 11: does any of it help? With and without the tools

Same question as in [09_04](../09_advanced_grounded_answers/09_04_with_and_without.ipynb), same method: one model, the same questions, twice.

```
uv run python scripts/benchmark_domain.py --model sonnet
```

For each of the 39 messages the script calls Claude through the Claude Code command line, with the same instructions both times (who it is, the four reasons to hand over, and a fixed last line).

| Run | What Claude has | Its last line |
|---|---|---|
| plain | nothing but the message | `SOURCE: none`, `UNKNOWN`, or `ESCALATE: <reason>` |
| tools | this notebook's MCP server, and no other tool | `SOURCE: easy-access#4`, `UNKNOWN`, or `ESCALATE: <reason>` |

In the tools run this is real tool use, not passages pasted in by us. Claude decides which tool to call and with what words, and the script reads the audit file of step 10 to see what it did.

The scores, for the 29 messages that have an answer: **right** means the reply holds the labelled key facts (`2.85`, `109`, `next working day`…), and **right source** that it also cites the labelled section or rate row. **Wrong** means it stated an answer that fails that check: an invented fact. For the 10 others, the only score is whether it handed over.

Like the benchmark of chapter 9, the script isn't part of the automatic checks. It uses your Claude account, and you don't need to run it.

### ❓ What came out?

This is one recorded run, not a number to quote:

| | |
|---|---|
| Date | 24 November 2026 |
| Model | `sonnet` (the alias, through Claude Code 2.1.316) |
| Messages | all 39 |
| Command | `uv run python scripts/benchmark_domain.py --model sonnet` |

| | plain (no tools) | with the three tools |
|---|---|---|
| **29 questions that have an answer** | | |
| right | 2 | **29** |
| right, and with the right source | 0 | **29** |
| wrong: an invented fact | 0 | 0 |
| no answer | 27 | 0 |
| **10 messages that must go to a person** | | |
| handed over | 10 | 10 |

So 0 answers with a source became 29 of 29, and nothing was invented in either run.

**The plain run was honest.** 27 times Claude said it had no information on Example Bank, and it's right: it has none. I expected a few confident guesses and got zero. The 2 it got right are both about ISA transfers, answered from what it knows about ISAs in general, with a note that it couldn't confirm this bank's own terms. A weaker model may behave worse here, and that's the first thing I'd test before choosing one.

**The tools run searched better than our own test did.** In step 4 the right section was in the first 3 for 19 of 22 questions. Claude got all 22. For question 2, the one about taking money out *before the year is up*, it didn't send the customer's words. It wrote its own query, and the audit file shows `fixed-1-year#4` coming back first. For two other questions it wasn't happy with the first result and searched again, once and twice more. It's what the tool's description asks for, and it's the first item on the list at the end of 09_04, measured this time.

**All 7 rates were exact,** each with its row, and each from one call of `get_rate` with the right day. That includes both fixed savers of step 6.

**All 10 handovers happened, in both runs.** That includes the three our word list missed. So for this, the instructions did the work and the tool only recorded it. The model read *I have had enough* the way a person would.

### ❓ So is it done? What this run doesn't show

No. A clean table on 39 messages is where the work starts.

- **I wrote the documents, the questions and the labels.** They fit each other better than real ones do. Real customers ask about things no sheet covers, three things at once, and in the middle of a story.
- **39 is small.** One message is about 2.6 points. *29 of 29* tells you nothing is badly broken. It can't tell 99% from 90%.
- **The handover messages are the easy kind.** Ten clear cases, with the rules spelled out in the prompt. The hard ones are a calm question with one half-sentence about a hospital stay in the middle. [10_02](10_02_spotting_complaints.ipynb) looks at those.
- **Nobody attacked it.** No message tried to talk the assistant out of its rules. [10_04](10_04_pitfalls_and_next_steps.ipynb) comes back to that.
- **One model, one day.** The `sonnet` alias will point at another model before long, and then this table is history. That's the frozen-model argument of 10_00 from the other side.

The table is what I'd show first. The list under it is what the conversation should then be about.

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why not put the rates into the sheets and let the search find them?

Then every rate change means editing a document, and an old copy of the sheet quotes an old rate with full confidence. A table with dates has one row per fact, and the tool can be tested: `tests/test_domain.py` checks the day a rate changes and the day before.

### ❓ Could the model answer AND hand over?

It could, and some firms want that: give the bereaved customer the phone number and pass the case on. That's a rule for the firm to write. I kept it strict here, because *answer nothing and hand over* is the easiest rule to test.

### ❓ Is the guard worth having if the model caught everything?

I'd keep it. The model caught 10 of 10 today, with this prompt. The guard catches its 7 whatever the prompt, the model or the day. When two layers fail in different ways, you want both.

### ❓ Does this make the assistant compliant with the rules?

No, and no test in a notebook could show that. It shows that on 39 invented messages the assistant cited its sources and handed over when told to. Whether a system meets a rule is for the firm and its compliance people to decide, with evidence from real use.

## 5. Recap

- One section is one passage, and it carries its sheet's version and date. Putting the sheet's title in front took recall@3 from 15 to 19 of 22.
- An exact number comes from a lookup tool with a date, never from a search and never from memory. When there's no row, the tool says so.
- Handing over has two layers: the model, told when in the tool's description, and a word-list guard under it that caught 7 of 10.
- Every tool call leaves an audit line: when, which tool, which data, what came back.
- In one run, Claude went from 0 to 29 of 29 sourced answers with the tools, invented nothing either way, and handed over all 10. On invented data, 39 messages.

**You can now:**

- search a company's documents and cite section and version
- say when a lookup tool beats a search: exact numbers that change with the date
- put a rule under the model for the cases that must go to a person

**Next:** [10_02: Spotting complaints and vulnerable customers in email](10_02_spotting_complaints.ipynb).